## Run ADLS OAuth setup

Run the ADLS OAuth notebook so Spark can access the storage account.

In [ ]:
%run ./adls_oauth

## Read raw CSV source folder

Read the `Parties` CSV folder directly from ADLS.

The CSV files do not contain headers, so Spark initially returns generic column names such as `_c0`, `_c1`, and `_c2`.

## Read Service Principal Credentials

This cell reads credentials again for the CDM connector test.

In [ ]:
# # Read credentials from secret scope.

# service_credential = dbutils.secrets.get(scope="kv-oncom-dev-scope",key="client-secret")
# appid = dbutils.secrets.get(scope="kv-oncom-dev-scope",key="app-id")
# tenantid = dbutils.secrets.get(scope="kv-oncom-dev-scope",key="tenant-id")

## StructType Fallback Test For Parties

This cell manually defines the schema for `Parties` based on `Parties.cdm.json`.

This proves that we can still read CDM-style data without the CDM connector.

## Define Parties Schema

This schema was verified from `Parties.cdm.json`.

The CSV files are headerless, so the column order in this schema must match the column order in the source CSV.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

# Define schema for the Parties entity.
# This schema is based on Parties.cdm.json.

partiesSchema = StructType([
    StructField("PartyId", LongType(), True),
    StructField("PartyName", StringType(), True),
    StructField("LastProcessedChange_DateTime", TimestampType(), True),
    StructField("DataLakeModified_DateTime", TimestampType(), True),
    StructField("PartyAddressCode", LongType(), True),
    StructField("EstablishedDate", TimestampType(), True),
    StructField("PartyEmailId", StringType(), True),
    StructField("PartyContactNumber", StringType(), True),
    StructField("RecordId", LongType(), True),
    StructField("TaxId", StringType(), True)
])

## Read Parties CSV With StructType Schema

This reads the headerless CSV folder using the manually defined schema.

This produces the same useful result that the CDM connector was supposed to produce.

In [ ]:
# Read Parties CSV files using the verified StructType schema.

dfParties = (spark.read.format("csv")
 .schema(partiesSchema)
 .option("header", "false")
 .option("path", "abfss://oaonoperationsdev@stoncomdev001.dfs.core.windows.net/oaon-sandbox.operations.dynamics.com/Tables/Purchase/Parties/")
 .load())

display(dfParties)

# Verify Parties Schema
#This verifies that the DataFrame has the expected column names and Spark data types.

dfParties.printSchema()